# Day 179 — Output Parsers & Structured Extraction
**Month 10 | LangChain + MLflow + Evidently**

| | |
|---|---|
| Dataset | ReviewPulse India — 600 rows, seed=155 |
| Split | Reference: rows 0–399 \| Current: rows 400–599 |
| LLM | Groq free API — llama-3.1-8b-instant |
| Total Points | 90 pts + 10★ bonus |

---

## What & Why

In production LLM pipelines, raw text output is almost never sufficient. You need **structured, machine-readable data** — JSON dicts, typed objects, validated fields — that downstream code can consume reliably. LangChain's Output Parsers sit between the LLM response and your application logic, converting free-form text into typed Python objects.

| Parser | Use Case | Reliability |
|--------|----------|-------------|
| `CommaSeparatedListOutputParser` | Tag extraction, keyword lists | Low — LLM may add spaces or extra commas |
| `StructuredOutputParser` | Multi-field dict from schema | Medium — format instructions injected into prompt |
| `PydanticOutputParser` | Typed, validated Python objects | High — Pydantic validates field types |
| `JsonOutputParser` | Arbitrary JSON objects | High — robust to markdown fences |
| `RetryOutputParser` | Fallback when parsing fails | Production essential |

**Real-world framing:** A client's review moderation pipeline receives 500 reviews/day. Manual tagging costs ₹15,000/month. A structured extraction pipeline costs ~₹800/month in API calls and runs in seconds.

---

## ⚙️ Cell 0 — Install & Imports (Run First — Then Restart Runtime)

In [1]:
# ── Pinned stack — mandatory for Month 10 ─────────────────────────────────────
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-groq==0.1.9 \
    groq==0.9.0 \
    httpx==0.27.0 \
    pydantic==2.7.4

print("Install complete — RESTART RUNTIME NOW, then run from Cell 1")

Install complete — RESTART RUNTIME NOW, then run from Cell 1


---
## 📦 Cell 1 — Imports & LLM Setup

In [2]:
import os
import json
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field, field_validator
from typing import Literal

from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.output_parsers import (
    CommaSeparatedListOutputParser,
    StructuredOutputParser,
    ResponseSchema,
    RetryOutputParser,
)
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser

# ── Groq API from Colab secret ─────────────────────────────────────────────────
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)
print("LLM ready:", llm.model_name)

LLM ready: llama-3.1-8b-instant


---
## 🗂️ Cell 2 — Raw Data (DO NOT MODIFY THIS CELL)

In [3]:
# ── RAW DATA — DO NOT MODIFY ───────────────────────────────────────────────────
np.random.seed(155)
n = 600

categories = {
    'positive': [
        'Excellent work, delivered ahead of schedule and exceeded all expectations.',
        'Good communication throughout the project, satisfied with the results.',
        'Delivered the project on time with high quality, will hire again.',
        'Outstanding freelancer, highly recommend for complex data projects.'
    ],
    'neutral': [
        'Average performance, met basic requirements but nothing outstanding.',
        'Some delays but the final output was acceptable.',
        'Neutral experience, work was done but lacked attention to detail.'
    ],
    'negative': [
        'Poor communication and missed multiple deadlines, very disappointed.',
        'Did not meet the agreed specifications, had to redo most of the work.',
        'Disappointed with the quality, did not meet the agreed specifications at all.'
    ]
}

sentiments = np.random.choice(['positive','neutral','negative'], size=n, p=[0.35,0.30,0.35])
ratings, texts = [], []
for s in sentiments:
    texts.append(np.random.choice(categories[s]))
    if s == 'positive':
        ratings.append(round(np.random.uniform(3.5, 5.0), 1))
    elif s == 'neutral':
        ratings.append(round(np.random.uniform(2.5, 3.5), 1))
    else:
        ratings.append(round(np.random.uniform(1.0, 2.5), 1))

df = pd.DataFrame({
    'review_id':     range(1, n+1),
    'freelancer_id': np.random.randint(1001, 1201, size=n),
    'review_text':   texts,
    'sentiment':     sentiments,
    'rating':        ratings,
    'hired_again':   np.random.choice(['Yes','No'], size=n, p=[0.36, 0.64]),
    'review_date':   pd.date_range('2023-01-01', periods=n, freq='D').strftime('%Y-%m-%d')
})

reference = df.iloc[:400].reset_index(drop=True)
current   = df.iloc[400:].reset_index(drop=True)

print(f"Full dataset: {len(df)} rows")
print(f"Reference split: {len(reference)} rows | Current split: {len(current)} rows")
print("\nCurrent split sentiment distribution:")
print(current['sentiment'].value_counts())
print(f"\nCurrent avg rating: {current['rating'].mean():.2f}")

Full dataset: 600 rows
Reference split: 400 rows | Current split: 200 rows

Current split sentiment distribution:
sentiment
negative    70
positive    66
neutral     64
Name: count, dtype: int64

Current avg rating: 2.97


---
## 📚 Cell 3 — Concept Notes

### Output Parsers — How They Work

Every LangChain output parser has two responsibilities:

1. **`get_format_instructions()`** — injects parsing instructions into the prompt so the LLM knows what format to return
2. **`parse(text)`** — converts the LLM's string response into a Python object

```
Prompt + format_instructions → LLM → raw string → parser.parse() → Python object
```

### Why Output Parsing Fails (and how to fix it)

| Failure Mode | Cause | Fix |
|---|---|---|
| Markdown fences in JSON | LLM wraps output in ```json ... ``` | Use `JsonOutputParser` (strips fences) |
| Wrong field names | LLM hallucinated key names | Use `PydanticOutputParser` (validates schema) |
| Extra commentary | LLM adds "Sure, here's the output:" | Explicit instruction: "Return ONLY the JSON" |
| Type mismatch | `"rating": "4"` instead of `"rating": 4` | Pydantic field type annotation enforces type |

### The LCEL Pattern (used in every task today)
```python
chain = prompt | llm | parser
result = chain.invoke({"text": "..."})
```

### `RetryOutputParser` — Production Guard
```python
retry_parser = RetryOutputParser.from_llm(parser=base_parser, llm=llm)
# If base_parser.parse() fails, retry_parser sends the bad output back to LLM
# with instructions to fix it — automated self-healing
```

---
## ✅ Task 1 — CommaSeparatedListOutputParser (15 pts)

**Goal:** Build a keyword-tag extractor for freelancer reviews.

**Steps:**
1. Instantiate `CommaSeparatedListOutputParser`
2. Build a `PromptTemplate` that:
   - Takes variable `{review}` and `{format_instructions}`
   - Instructs the LLM: *"Extract 3–5 keyword tags that describe this freelancer review. Tags must be lowercase, single words or short hyphenated phrases."*
3. Build the LCEL chain: `prompt | llm | parser`
4. Run on these 3 reviews (use `.review_text` from the current split, rows 0, 1, 2):
   - `review_a = current.iloc[0]['review_text']`
   - `review_b = current.iloc[1]['review_text']`
   - `review_c = current.iloc[2]['review_text']`
5. Print all 3 results. Each must be a **Python list** (not a string).
6. Print `type(tags_a)` to confirm it is `<class 'list'>`

**Grading:**
- Parser instantiated + `get_format_instructions()` injected into prompt: 4 pts
- LCEL chain built correctly: 3 pts
- All 3 reviews processed, output is list type: 5 pts
- Each result has 3–5 tags: 3 pts

In [8]:
# ── TASK 1 — CommaSeparatedListOutputParser ──────────────────────────────────
# Goal: Extract 3–5 keyword tags from freelancer reviews as a Python list.
# Method: Use ChatPromptTemplate with system instruction, inject format_instructions.

from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.prompts import ChatPromptTemplate

# 1. Instantiate parser
list_parser = CommaSeparatedListOutputParser()

# 2. Build chat prompt with system message
system_msg = (
    "You extract keyword tags from freelancer reviews. "
    "Return only a comma-separated list of 3-5 tags, nothing else. "
    "Do not include explanations or extra text."
)
human_msg = "Review: {review}\n{format_instructions}"
prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg)
])

# 3. Build LCEL chain
list_chain = prompt | llm | list_parser

# 4. Process reviews (rows 0,1,2 of current split)
review_a = current.iloc[0]['review_text']
review_b = current.iloc[1]['review_text']
review_c = current.iloc[2]['review_text']

tags_a = list_chain.invoke({"review": review_a, "format_instructions": list_parser.get_format_instructions()})
tags_b = list_chain.invoke({"review": review_b, "format_instructions": list_parser.get_format_instructions()})
tags_c = list_chain.invoke({"review": review_c, "format_instructions": list_parser.get_format_instructions()})

# 5. Print results
print("Tags for review A:", tags_a)
print("Tags for review B:", tags_b)
print("Tags for review C:", tags_c)
print("type(tags_a):", type(tags_a))   # Should be <class 'list'>

Tags for review A: ['delays', 'output', 'acceptable']
Tags for review B: ['neutral', 'attention to detail', 'lack of quality']
Tags for review C: ['unreliable', 'poor_communication', 'missed_deadlines']
type(tags_a): <class 'list'>


---
## ✅ Task 2 — StructuredOutputParser (20 pts)

**Goal:** Extract a structured 3-field dict from a single review.

**Steps:**
1. Define these 3 `ResponseSchema` objects:
   - `sentiment` — `"The sentiment of the review: positive, neutral, or negative"`
   - `rating_category` — `"Rate the review quality: low (rating < 2.5), medium (2.5–3.5), high (> 3.5)"`
   - `action` — `"Recommended platform action: flag, monitor, or approve"`
2. Build `StructuredOutputParser.from_response_schemas([...])`
3. Build a `PromptTemplate` with variables `{format_instructions}` and `{review}`
   - Inject the format instructions from `parser.get_format_instructions()`
4. Build LCEL chain: `prompt | llm | parser`
5. Run on **review_id = 401** (first row of `current` split)
   - `review_401 = current.iloc[0]['review_text']`
6. Print the full result dict
7. Print: `f"Review 401 → sentiment={result['sentiment']}, action={result['action']}"`

**Grading:**
- All 3 ResponseSchema objects correctly defined: 5 pts
- StructuredOutputParser built from schemas: 4 pts
- LCEL chain with format_instructions in prompt: 5 pts
- Output is a dict with all 3 keys present: 4 pts
- Print statement showing sentiment + action: 2 pts

In [9]:
# ── TASK 2 — StructuredOutputParser ──────────────────────────────────────────
# Goal: Extract a 3‑field dict (sentiment, rating_category, action) from a review.
# Method: Define ResponseSchemas, build parser, use ChatPromptTemplate with system.

from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.prompts import ChatPromptTemplate

# 1. Define response schemas
schemas = [
    ResponseSchema(name="sentiment",
                   description="The sentiment of the review: positive, neutral, or negative"),
    ResponseSchema(name="rating_category",
                   description="Rate the review quality: low (rating < 2.5), medium (2.5–3.5), high (> 3.5)"),
    ResponseSchema(name="action",
                   description="Recommended platform action: flag, monitor, or approve")
]

# 2. Build parser
structured_parser = StructuredOutputParser.from_response_schemas(schemas)

# 3. Build chat prompt
system_msg = (
    "You analyse freelancer reviews and return a JSON object with the specified fields. "
    "Output only the JSON object, no extra text, explanations, or code."
)
human_msg = "Review: {review}\n{format_instructions}"
prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg)
])

# 4. Build LCEL chain
structured_chain = prompt | llm | structured_parser

# 5. Run on review_id = 401 (first row of current)
review_401 = current.iloc[0]['review_text']
result_struct = structured_chain.invoke({
    "review": review_401,
    "format_instructions": structured_parser.get_format_instructions()
})

# 6. Print full dict
print("Structured result:", result_struct)

# 7. Print sentiment and action
print(f"Review 401 → sentiment={result_struct['sentiment']}, action={result_struct['action']}")

Structured result: {'sentiment': 'neutral', 'rating_category': 'medium', 'action': 'monitor'}
Review 401 → sentiment=neutral, action=monitor


---
## ✅ Task 3 — PydanticOutputParser (20 pts)

**Goal:** Define a typed Pydantic model and extract validated structured data.

**Steps:**
1. Define a Pydantic `BaseModel` called `ReviewInsight` with these fields:
   ```python
   sentiment:        Literal['positive', 'neutral', 'negative']
   confidence:       float   # 0.0 to 1.0
   key_issue:        str     # one-sentence summary of the main issue
   escalate:         bool    # True if the review needs immediate attention
   ```
2. Instantiate `PydanticOutputParser(pydantic_object=ReviewInsight)`
3. Build a `PromptTemplate` that:
   - Instructs the LLM to analyse the review and return a `ReviewInsight` object
   - Injects `{format_instructions}` from the parser
   - Has variable `{review}`
4. Build LCEL chain: `prompt | llm | parser`
5. Run on **review_id = 403** (third row of `current`, index 2):
   - `review_403 = current.iloc[2]['review_text']`
6. Print the parsed `ReviewInsight` object
7. Print: `f"Escalate: {result.escalate} | Confidence: {result.confidence:.2f}"`
8. Print `type(result)` — must be `<class '__main__.ReviewInsight'>`

**Grading:**
- `ReviewInsight` model with all 4 fields and correct types: 6 pts
- `PydanticOutputParser` instantiated correctly: 4 pts
- LCEL chain with format_instructions injected: 4 pts
- Result is a `ReviewInsight` instance (not a dict): 4 pts
- Print statement showing `escalate` + `confidence`: 2 pts

In [16]:
# ── TASK 3 — PydanticOutputParser ────────────────────────────────────────────
# Goal: Extract typed and validated data using a Pydantic model (ReviewInsight).
# Method: Define model, use ChatPromptTemplate with system to force JSON.

from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate

# 1. Define Pydantic model
class ReviewInsight(BaseModel):
    sentiment: Literal['positive', 'neutral', 'negative']
    confidence: float = Field(..., ge=0.0, le=1.0)
    key_issue: str = Field(..., description="one-sentence summary of the main issue")
    escalate: bool = Field(..., description="True if review needs immediate attention")

# 2. Instantiate parser
pydantic_parser = PydanticOutputParser(pydantic_object=ReviewInsight)

# 3. Build chat prompt with system (store in pydantic_prompt)
system_msg = (
    "You are a helpful assistant that extracts structured data from reviews. "
    "Always output **only** a valid JSON object that matches the required schema. "
    "Do not include any extra text, explanations, or code. "
    "Only output the JSON object."
)
human_msg = "Review: {review}\n{format_instructions}"
pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg)
])

# 4. Build LCEL chain (use pydantic_prompt)
pydantic_chain = pydantic_prompt | llm | pydantic_parser

# 5. Run on review_id = 403 (third row of current, index 2)
review_403 = current.iloc[2]['review_text']
result_pydantic = pydantic_chain.invoke({
    "review": review_403,
    "format_instructions": pydantic_parser.get_format_instructions()
})

# 6. Print parsed object
print("Parsed ReviewInsight:", result_pydantic)

# 7. Print escalate and confidence
print(f"Escalate: {result_pydantic.escalate} | Confidence: {result_pydantic.confidence:.2f}")

# 8. Confirm type
print("type(result_pydantic):", type(result_pydantic))   # Should be <class '__main__.ReviewInsight'>

Parsed ReviewInsight: sentiment='negative' confidence=0.0 key_issue='Did not meet the agreed specifications, had to redo most of the work.' escalate=True
Escalate: True | Confidence: 0.00
type(result_pydantic): <class '__main__.ReviewInsight'>


---
## ✅ Task 4 — JsonOutputParser + Batch Processing (20 pts)

**Goal:** Process 10 reviews in a batch, extract structured JSON per review, count flagged items.

**Flagging rule:** A review is flagged if `rating < 2.5`.

**Steps:**
1. Instantiate `JsonOutputParser()`
2. Build a `PromptTemplate` that:
   - Takes `{review}` and `{rating}`
   - Instructs: *"Analyse this freelancer review. Return a JSON object with keys: `sentiment` (positive/neutral/negative), `flag` (true if the review describes serious problems, false otherwise), `reason` (one sentence why you flagged or approved)."*
   - Injects `{format_instructions}` from `parser.get_format_instructions()`
3. Build LCEL chain: `prompt | llm | parser`
4. Process **rows 0–9 of the `current` split** (review_ids 401–410) in a for-loop:
   - For each row, call `chain.invoke({'review': row.review_text, 'rating': row.rating, 'format_instructions': ...})`
   - Store results in a list called `batch_results`
   - Add `review_id` and `actual_rating` to each result dict before appending
5. Convert `batch_results` to a DataFrame called `results_df`
6. Compute `flagged_count = results_df['flag'].sum()` — **print this**
7. Print the full `results_df` (all 10 rows)

**Grading:**
- `JsonOutputParser` used correctly (not `json.loads` on raw string): 4 pts
- All 10 rows processed in loop: 4 pts
- Each result dict has `sentiment`, `flag`, `reason`, `review_id`, `actual_rating`: 5 pts
- `results_df` built from list: 4 pts
- `flagged_count` printed: 3 pts

**Pre-computed anchor (ratings, not LLM logic):**
- Rows with `rating < 2.5` in the first 10 of current: **review_ids 403, 405, 408** → `actual_rating` flags = 3
- Your LLM's `flag` values may differ (it uses text reasoning, not the numeric threshold). That's expected — both are printed for comparison.

In [11]:
# ── TASK 4 — JsonOutputParser + Batch Processing ─────────────────────────────
# Goal: Process 10 reviews in batch, extract JSON with sentiment, flag, reason,
#       and compare LLM flagging with rating-based threshold.
# Method: Use JsonOutputParser with ChatPromptTemplate, loop over rows 0-9.

from langchain_core.output_parsers import JsonOutputParser
from langchain.prompts import ChatPromptTemplate
import pandas as pd

# 1. Instantiate JsonOutputParser
json_parser = JsonOutputParser()

# 2. Build chat prompt with system
system_msg = (
    "You analyse freelancer reviews and return a JSON object with exactly these keys: "
    "'sentiment' (positive/neutral/negative), "
    "'flag' (true if the review describes serious problems, false otherwise), "
    "'reason' (one sentence why you flagged or approved). "
    "Output only the JSON object, no extra text."
)
human_msg = "Review: {review}\nRating: {rating}\n{format_instructions}"
prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg)
])

# 3. Build LCEL chain
json_chain = prompt | llm | json_parser

# 4. Process rows 0–9 of current split
batch_results = []
for i in range(10):
    row = current.iloc[i]
    result = json_chain.invoke({
        "review": row['review_text'],
        "rating": row['rating'],
        "format_instructions": json_parser.get_format_instructions()
    })
    result['review_id'] = row['review_id']
    result['actual_rating'] = row['rating']
    batch_results.append(result)

# 5. Convert to DataFrame
results_df = pd.DataFrame(batch_results)

# 6. Compute flagged_count (LLM flag = True)
flagged_count = results_df['flag'].sum()
print(f"Flagged count (LLM-based): {flagged_count}")

# 7. Print full DataFrame
print("\nBatch results (rows 0-9 of current split):")
print(results_df)

Flagged count (LLM-based): 3

Batch results (rows 0-9 of current split):
  sentiment   flag                                             reason  \
0   neutral  False  The review is neutral because it mentions both...   
1   neutral  False  The review is neutral because it mentions both...   
2  negative   True  The freelancer failed to meet the agreed speci...   
3   neutral  False  The review is neutral because it mentions both...   
4  negative   True  The freelancer failed to meet the agreed speci...   
5  positive  False  The review is overwhelmingly positive and does...   
6  positive  False  The review is overwhelmingly positive and does...   
7  negative   True  The freelancer had serious problems with commu...   
8  positive  False  The review is overwhelmingly positive and does...   
9  positive  False  The review is generally positive and does not ...   

   review_id  actual_rating  
0        401            3.1  
1        402            2.6  
2        403            2.4  
3  

---
## ✅ Task 5 — NRA Business Insight (15 pts)

**Goal:** Use the structured extraction results to write a business insight in NRA format.

**Context:** You are presenting output parser pipeline results to a freelance platform client.

**Anchor stats from current split (rows 400–599):**
- `avg_rating = 2.97`
- `negative% = 35.00%`
- `positive% = 33.00%`
- `hired_again_yes% = 36.00%`

**Steps:**
1. Print the anchor stats above (read them from computed DataFrame output — do not type from memory)
2. Write a **markdown cell** with 3 NRA bullets. Each bullet must:
   - Start with `**[Title]:**` that states the finding with a number
   - `Number` — single stat anchored to computed output
   - `Reason` — causal mechanism (not a description of the stat)
   - `Action` — specific, committed, names concrete parameters or models

**Grading:**
- Anchor stats printed from computed values (not typed from memory): 3 pts
- 3 NRA bullets present, each with title + N + R + A: 6 pts
- Number is a single anchor stat per bullet: 3 pts
- Action is specific and committed (no hedging): 3 pts

In [12]:
# ── TASK 5A — Print anchor stats from computed DataFrame ──────────────────────
avg_rating       = round(current['rating'].mean(), 2)
pct_negative     = round((current['sentiment'] == 'negative').mean() * 100, 2)
pct_positive     = round((current['sentiment'] == 'positive').mean() * 100, 2)
pct_hired_again  = round((current['hired_again'] == 'Yes').mean() * 100, 2)

print(f"avg_rating      = {avg_rating}")
print(f"negative%       = {pct_negative}%")
print(f"positive%       = {pct_positive}%")
print(f"hired_again Yes = {pct_hired_again}%")

avg_rating      = 2.97
negative%       = 35.0%
positive%       = 33.0%
hired_again Yes = 36.0%


### Task 5B — NRA Insight

**[35% Negative Reviews]:**
- **Number:** 35.0% of reviews are classified as negative.
- **Reason:** Clients who experience poor communication, missed deadlines, or unmet specifications are more likely to leave negative feedback, which directly erodes platform trust and reduces repeat business.
- **Action:** We will deploy the sentiment classification pipeline on all incoming reviews and automatically route every new negative review to a priority queue. A support agent will initiate client outreach within **1 hour** to resolve issues and recover trust.

**[Average Rating of 2.97]:**
- **Number:** The average rating across the current batch is 2.97 out of 5.0.
- **Reason:** A sub‑3.0 average rating signals a systemic quality gap—freelancers are failing to meet client expectations, particularly in meeting specifications and deadlines.
- **Action:** We will implement a mandatory quality review for any freelancer with an average rating < 3.0 and more than 3 negative reviews in the last 30 days. Such freelancers will be **suspended** until they complete a retraining module on communication and specification adherence.

**[36% Re‑hire Rate]:**
- **Number:** Only 36.0% of clients indicate they would hire the freelancer again.
- **Reason:** Low re‑hire intention indicates that even when sentiment is neutral, the final deliverable often falls short of expectations, creating a silent churn risk.
- **Action:** We will enrich the output parser to extract the `key_issue` (as in Task 3) for every negative or neutral review and automatically generate a structured summary. This summary will be sent to the freelancer as part of a **performance improvement plan** within 48 hours, with clear corrective actions.

---
## ⭐ Bonus Task — RetryOutputParser (10★)

**Goal:** Implement a production-grade self-healing parser that recovers from malformed LLM output.

**Context:** In production, LLMs occasionally return invalid JSON or miss required fields. `RetryOutputParser` automatically re-prompts the LLM with the broken output and asks it to fix itself.

**Steps:**
1. Reuse the `PydanticOutputParser(pydantic_object=ReviewInsight)` from Task 3
2. Define a deliberately malformed response string:
   ```python
   bad_response = '{"sentiment": "neg", "confidence": 0.9}'  
   # Missing 'key_issue' and 'escalate'; 'neg' is not a valid Literal
   ```
3. Confirm it fails: wrap `pydantic_parser.parse(bad_response)` in a try/except and print the error message
4. Build `RetryOutputParser.from_llm(parser=pydantic_parser, llm=llm)`
5. Use `retry_parser.parse_with_prompt(bad_response, prompt_value)` where `prompt_value` is the formatted prompt from Task 3 for review_id 403
6. Print the recovered `ReviewInsight` object
7. Print: `"Self-healed: True"` if `isinstance(result, ReviewInsight)` else `"Self-healed: False"`

**Grading (bonus):**
- `bad_response` defined and parse failure caught + printed: 3★
- `RetryOutputParser` instantiated with `from_llm`: 3★
- Recovery successful, result is `ReviewInsight` instance: 4★

In [17]:
# ── BONUS — RetryOutputParser ──────────────────────────────────────────────────
# Goal: Demonstrate self‑healing parsing using RetryOutputParser.
# Method: Reuse the Pydantic parser from Task 3, define a malformed JSON,
#         catch parse error, then use RetryOutputParser to fix it.

from langchain.output_parsers import RetryOutputParser

# 1. Format the prompt with the review text and format instructions using pydantic_prompt
prompt_value = pydantic_prompt.format_prompt(
    review=review_403,
    format_instructions=pydantic_parser.get_format_instructions()
)

# 2. Deliberately malformed response
bad_response = '{"sentiment": "neg", "confidence": 0.9}'   # missing key_issue & escalate, invalid Literal

# 3. Confirm failure with try/except
print("Attempting to parse bad_response with pydantic_parser...")
try:
    failed_parse = pydantic_parser.parse(bad_response)
except Exception as e:
    print(f"Parse error (as expected): {e}")

# 4. Build RetryOutputParser
retry_parser = RetryOutputParser.from_llm(parser=pydantic_parser, llm=llm)

# 5. Use parse_with_prompt to recover
recovered = retry_parser.parse_with_prompt(bad_response, prompt_value)

# 6. Print recovered object
print("\nRecovered ReviewInsight:")
print(recovered)

# 7. Self-healed check
self_healed = isinstance(recovered, ReviewInsight)
print(f"Self-healed: {self_healed}")   # Should be True

Attempting to parse bad_response with pydantic_parser...
Parse error (as expected): Failed to parse ReviewInsight from completion {"sentiment": "neg", "confidence": 0.9}. Got: 3 validation errors for ReviewInsight
sentiment
  Input should be 'positive', 'neutral' or 'negative' [type=literal_error, input_value='neg', input_type=str]
    For further information visit https://errors.pydantic.dev/2.7/v/literal_error
key_issue
  Field required [type=missing, input_value={'sentiment': 'neg', 'confidence': 0.9}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.7/v/missing
escalate
  Field required [type=missing, input_value={'sentiment': 'neg', 'confidence': 0.9}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.7/v/missing

Recovered ReviewInsight:
sentiment='negative' confidence=0.9 key_issue='Did not meet the agreed specifications, had to redo most of the work.' escalate=True
Self-healed: True


---
## 📊 Scoring Rubric

| Task | Topic | Points | Breakdown |
|------|-------|--------|-----------|
| T1 | CommaSeparatedListOutputParser | 15 | Parser instantiated + format_instructions injected: 4 \| LCEL chain correct: 3 \| All 3 reviews processed as list: 5 \| 3–5 tags per result: 3 |
| T2 | StructuredOutputParser | 20 | 3 ResponseSchema objects correct: 5 \| Parser built from schemas: 4 \| LCEL with format_instructions: 5 \| Dict with all 3 keys: 4 \| Print statement: 2 |
| T3 | PydanticOutputParser | 20 | ReviewInsight with 4 correct fields/types: 6 \| Parser instantiated: 4 \| LCEL with format_instructions: 4 \| Result is ReviewInsight instance: 4 \| Print statement: 2 |
| T4 | JsonOutputParser + Batch | 20 | JsonOutputParser used (not json.loads): 4 \| All 10 rows processed: 4 \| All 5 columns present: 5 \| results_df built from list: 4 \| flagged_count printed: 3 |
| T5 | NRA Business Insight | 15 | Anchor stats printed from DataFrame: 3 \| 3 NRA bullets with title+N+R+A: 6 \| Single anchor stat per Number: 3 \| Specific committed Actions: 3 |
| ★ Bonus | RetryOutputParser | 10★ | bad_response defined + parse failure caught: 3★ \| RetryOutputParser with from_llm: 3★ \| Recovery successful, ReviewInsight instance: 4★ |
| **Total** | | **90 + 10★** | |

---

### ⚠️ Automatic Deductions

| Violation | Penalty |
|-----------|--------|
| Using `eval()` instead of a parser | −5 pts per instance |
| Accessing `.content` directly instead of parsing through chain | −5 pts |
| `format_instructions` hard-coded instead of from `parser.get_format_instructions()` | −4 pts |
| NRA Number is two stats instead of one anchor | −2 pts per bullet |
| NRA Action uses hedging language ("could", "might", "consider") | −2 pts per bullet |
| NRA stat typed from memory instead of read from cell output | −3 pts |
| `type(result)` is dict instead of ReviewInsight in T3 | −4 pts |

---
## 🎤 Interview Answer

**Q: When would you use `PydanticOutputParser` over `StructuredOutputParser`?**

> *"StructuredOutputParser works well for simple multi-field extraction when I just need a dict and can tolerate some inconsistency — it's quick to set up and fine for prototyping. PydanticOutputParser is the production choice because Pydantic enforces field types at parse time: a `confidence` field typed as `float` raises a `ValidationError` immediately if the LLM returns `'high'`, rather than passing silently and crashing downstream logic. The `Literal` type constraint on `sentiment` is especially powerful — it guarantees the output is exactly one of `positive`, `neutral`, or `negative` with zero LLM creativity. In practice I layer `RetryOutputParser` on top as a fallback: one self-healing retry catches most transient formatting errors without needing a full circuit breaker, which keeps the pipeline running rather than crashing on a single bad response."*

---

**GitHub commit when done:**
```
feat: Day179 - Output Parsers & Structured Extraction [score/90+bonus★]
```